# 03. 기준선 모델

1~7일 뒤의 일별 판매량을 예측하기 전에, 각 목표일과 같은 요일의 직전 주 판매량이 어느 정도 성능을 내는지 확인합니다. 이 값은 모든 예측 거리에서 기준일 이전에 관측되므로 안전하게 사용할 수 있습니다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw"
OUTPUT_PATH = PROJECT_ROOT / "outputs"

In [ ]:
def rmsle(actual, predicted):
    actual = np.clip(np.asarray(actual), 0, None)
    predicted = np.clip(np.asarray(predicted), 0, None)
    return np.sqrt(np.mean((np.log1p(actual) - np.log1p(predicted)) ** 2))


def mae(actual, predicted):
    return np.mean(np.abs(np.asarray(actual) - np.asarray(predicted)))


def evaluate_baselines(frame, columns):
    rows = []
    for name, column in columns.items():
        rows.append({"model": name, "rmsle": rmsle(frame["target_sales"], frame[column]), "mae": mae(frame["target_sales"], frame[column])})
    return pd.DataFrame(rows).set_index("model").sort_values("rmsle")

## 기준선 정의

| 기준선 | 사용 정보 | 활용 목적 |
|---|---|---|
| 직전 주 일별 패턴 | 목표일보다 7일 전의 동일 점포·상품군 판매량 | 가장 최근 주간 패턴 반영 |
| 동일 요일 최근 4주 평균 | 목표일보다 7·14·21·28일 전 판매량의 평균 | 한 주의 이상치 영향 완화 |
| LightGBM | 과거 수요·예측 거리·목표일 사전 정보 | 주간 기준선 대비 개선 확인 |

예측 거리가 1~7일이므로 목표일보다 7일 전 날짜는 항상 기준일과 같거나 이전입니다. 목표일 직전 판매량처럼 예측 시점에 알 수 없는 값은 사용하지 않습니다.

In [ ]:
predictions_path = OUTPUT_PATH / "validation_predictions_lightgbm.csv"
predictions_path = predictions_path if predictions_path.exists() else OUTPUT_PATH / "validation_predictions.csv"
predictions = pd.read_csv(
    predictions_path,
    parse_dates=["date", "target_date"],
)

In [ ]:
baseline_columns = {
    "LightGBM": "prediction",
    "Previous-week daily profile": "sales_same_weekday_last_week",
    "Same-weekday 4-week mean": "sales_same_weekday_4w_mean",
}
baseline_scores = evaluate_baselines(predictions, baseline_columns)
baseline_scores.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
baseline_scores["rmsle"].sort_values().plot.barh(ax=axes[0], title="RMSLE")
baseline_scores["mae"].sort_values().plot.barh(ax=axes[1], color="tab:orange", title="MAE")
axes[0].set(xlabel="lower is better", ylabel="")
axes[1].set(xlabel="lower is better", ylabel="")
plt.tight_layout()

전체 성능뿐 아니라 `forecast_horizon`별 RMSLE와 MAE를 함께 확인합니다. 예측 거리가 길어질수록 오차가 증가하는지, 통합 LightGBM이 각 거리에서 주간 계절 기준선을 안정적으로 개선하는지가 핵심 판단 기준입니다.

In [ ]:
horizon_scores = []
for horizon, group in predictions.groupby("forecast_horizon"):
    for name, column in baseline_columns.items():
        horizon_scores.append({
            "forecast_horizon": horizon,
            "model": name,
            "rmsle": rmsle(group["target_sales"], group[column]),
            "mae": mae(group["target_sales"], group[column]),
        })
horizon_scores = pd.DataFrame(horizon_scores)
horizon_scores.pivot(index="forecast_horizon", columns="model", values=["rmsle", "mae"]).round(4)